# CLEAN WIDE FILE IN POLISH

In [1]:
import pandas as pd
import numpy as np
import re

INPUT_FILE = "2026 Data.xlsx"
SHEET = "2019-2024"
OUTPUT_FILE = "Data_clean_2019-2024.xlsx"

# -------------------------
# 1) Read raw (no header)
# -------------------------
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET, header=None, dtype=str)

indicator_row = raw.iloc[0].tolist()
year_row = raw.iloc[1].tolist()

def normalise_year(y) -> str:
    """Turn '2 020' or '2020.0' into '2020' if possible."""
    if pd.isna(y):
        return ""
    s = str(y).strip()
    s = s.replace("\u00a0", " ").strip()     # NBSP to space
    s = re.sub(r"\s+", "", s)                # remove all spaces
    s = re.sub(r"\.0$", "", s)               # drop trailing .0
    return s if re.fullmatch(r"\d{4}", s) else ""

# Build final column names using 2-row header logic
final_cols = []
for ind, yr in zip(indicator_row, year_row):
    ind_s = "" if pd.isna(ind) else str(ind).strip()
    yr_s = normalise_year(yr)

    if yr_s:  # metric__year
        name = ind_s if ind_s else "unknown_metric"
        final_cols.append(f"{name}__{yr_s}")
    else:     # identifier column
        final_cols.append(ind_s if ind_s else f"col_{len(final_cols)}")

# -------------------------
# 2) Build dataframe (data starts at row 2)
# -------------------------
df = raw.iloc[2:].copy()
df.columns = final_cols
df = df.reset_index(drop=True)

# -------------------------
# 3) Clean numeric columns AFTER S/J
# -------------------------
if "S/J" not in df.columns:
    raise ValueError("Column 'S/J' not found after rebuilding headers. Check the header rows / first columns.")

missing_markers = ["b.d.", "b.d", "bd", "BD", "B.D.", "—", "-", ""]
df = df.replace(missing_markers, np.nan)

sj_idx = list(df.columns).index("S/J")
numeric_cols = list(df.columns)[sj_idx + 1 :]

def to_num(series: pd.Series) -> pd.Series:
    """
    Convert messy numeric strings to float.
    - handles Polish decimal comma
    - handles spaces & NBSP
    - coerces unparseable values to NaN
    """
    x = series.astype(str).str.strip()

    # standardise missing markers again (in case they appear as strings)
    x = x.replace(missing_markers + ["nan", "None"], np.nan)

    # remove spaces incl NBSP
    x = x.str.replace("\u00a0", "", regex=False).str.replace(" ", "", regex=False)

    # If both comma and dot are present -> remove commas (thousands sep)
    mask_both = (
        x.notna()
        & x.str.contains(",", regex=False, na=False)
        & x.str.contains(r"\.", regex=True, na=False)
    )
    x.loc[mask_both] = x.loc[mask_both].str.replace(",", "", regex=False)

    # If comma present and dot absent -> treat comma as decimal separator
    mask_comma_decimal = (
        x.notna()
        & x.str.contains(",", regex=False, na=False)
        & ~x.str.contains(r"\.", regex=True, na=False)
    )
    x.loc[mask_comma_decimal] = x.loc[mask_comma_decimal].str.replace(",", ".", regex=False)

    return pd.to_numeric(x, errors="coerce")

# Convert numeric columns
for c in numeric_cols:
    df[c] = to_num(df[c])

# -------------------------
# 4) Recommended audits
# -------------------------
print("Final shape:", df.shape)

# (A) Confirm numeric dtypes
print("\nNumeric dtypes after S/J:")
print(df[numeric_cols].dtypes.value_counts())

# (B) Any columns still not numeric?
non_numeric_cols = df[numeric_cols].select_dtypes(exclude=["number"]).columns.tolist()
print("\nColumns after S/J that are still non-numeric (should be empty):")
print(non_numeric_cols)

# (C) Missing counts after conversion (top 20)
missing_after = df[numeric_cols].isna().sum().sort_values(ascending=False)
print("\nMissing (NaN) counts in numeric columns after conversion (top 20):")
print(missing_after.head(20))

# (D) Sample columns to confirm header reconstruction
print("\nSample columns:")
print(list(df.columns[:12]))

# -------------------------
# 5) Save once at the end
# -------------------------
df.to_excel(OUTPUT_FILE, index=False)
print("\nSaved:", OUTPUT_FILE)

df_wide = df

Final shape: (2454, 99)

Numeric dtypes after S/J:
float64    79
Name: count, dtype: int64

Columns after S/J that are still non-numeric (should be empty):
[]

Missing (NaN) counts in numeric columns after conversion (top 20):
Zatrudnienie (średnioroczne w etatach)__2019    46
Zatrudnienie (średnioroczne w etatach)__2020    43
Zatrudnienie (średnioroczne w etatach)__2021    35
Przychody z eksportu__2019                      35
Wynagrodzenia (z narzutami)__2019               31
Przychody z eksportu__2020                      30
Zatrudnienie (średnioroczne w etatach)__2022    28
Przychody z eksportu__2021                      26
Zatrudnienie (średnioroczne w etatach)__2024    26
Zatrudnienie (średnioroczne w etatach)__2023    25
Przychody z eksportu__2024                      25
Przychody z eksportu__2022                      24
Amortyzacja__2019                               24
Wynagrodzenia z narzutami__2024                 22
Wynagrodzenia z narzutami__2020                 21
Wynagrod

# CLEAN LONG FILE IN ENGLISH

In [2]:
import pandas as pd
import numpy as np
import re

INPUT_FILE = "Data_clean_2019-2024.xlsx"
OUTPUT_XLSX = "Data_panel_2019-2024.xlsx"
OUTPUT_PARQUET = "../2026/Data_panel_2019-2024.parquet"

# --------------------------------------------------
# Helper
# --------------------------------------------------

def slugify(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("ą", "a").replace("ć", "c").replace("ę", "e").replace("ł", "l")
    s = s.replace("ń", "n").replace("ó", "o").replace("ś", "s").replace("ż", "z").replace("ź", "z")
    s = re.sub(r"[^\w]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

# --------------------------------------------------
# 1. Load wide data
# --------------------------------------------------

df = pd.read_excel(INPUT_FILE)

# --------------------------------------------------
# 2. Rename ranking columns 2019-2024 -> rank_YYYY
#    Keep them as separate columns in the final panel
# --------------------------------------------------

rank_years = [str(y) for y in range(2019, 2025)]
rename_rank = {y: f"rank_{y}" for y in rank_years if y in df.columns}
df = df.rename(columns=rename_rank)

rank_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("rank_")]

# --------------------------------------------------
# 3. Identify metric__year columns
# --------------------------------------------------

value_cols = [c for c in df.columns if isinstance(c, str) and "__" in c]

if not value_cols:
    raise ValueError("No metric__year columns found (expected '__' in column names).")

# Preserve metric order from wide format
metric_pl_in_order = []
seen = set()

for c in value_cols:
    metric_pl, _ = c.rsplit("__", 1)
    if metric_pl not in seen:
        metric_pl_in_order.append(metric_pl)
        seen.add(metric_pl)

# --------------------------------------------------
# 4. Metric translation map
# --------------------------------------------------

metric_map = {
    "Przychody ze sprzedaży": "sales",
    "Wynik operacyjny": "operating_result",
    "Wynik finansowy brutto": "profit_before_tax",
    "Podatek dochodowy": "income_tax",
    "Wynik finansowy netto": "net_profit",
    "Amortyzacja": "depreciation",
    "Przychody z eksportu (sprzedaż po zagranicę kraju)": "exports",
    "Przychody z eksportu": "exports",
    "Zatrudnienie (średnioroczne w etatach)": "employment",
    "Wynagrodzenia z narzutami": "wages_total",
    "Wynagrodzenia (z narzutami)": "wages_total",
    "Suma bilansowa": "total_assets",
    "Aktywa": "total_assets",
    "Aktywa trwałe": "fixed_assets",
    "Aktywa obrotowe": "current_assets",
    "Kapitały własne": "equity",
    "Zobowiązania_i_rezerwy_na_zobowiązania": "liabilities_provisions",
    "Zobowiązania razem": "total_liabilities",
}

metric_en_in_order = []
seen2 = set()

for m_pl in metric_pl_in_order:
    m_en = metric_map.get(m_pl, slugify(m_pl))
    if m_en not in seen2:
        metric_en_in_order.append(m_en)
        seen2.add(m_en)

# --------------------------------------------------
# 5. Pivot financial data to panel
#    Small improvement: keep only needed columns before melt
# --------------------------------------------------

if "NIP" not in df.columns:
    raise ValueError("Column 'NIP' not found.")

KEY = ["NIP"]

fin_source = df[KEY + value_cols].copy()

fin_long = fin_source.melt(
    id_vars=KEY,
    value_vars=value_cols,
    var_name="variable",
    value_name="value"
)

fin_long[["metric_pl", "year"]] = fin_long["variable"].str.rsplit("__", n=1, expand=True)
fin_long["year"] = pd.to_numeric(fin_long["year"], errors="coerce").astype("Int64")
fin_long = fin_long[fin_long["year"].notna()].copy()

fin_long["metric"] = fin_long["metric_pl"].map(metric_map).fillna(
    fin_long["metric_pl"].map(slugify)
)

fin_long["metric"] = pd.Categorical(
    fin_long["metric"],
    categories=metric_en_in_order,
    ordered=True
)

fin_panel = (
    fin_long
    .pivot_table(
        index=KEY + ["year"],
        columns="metric",
        values="value",
        aggfunc="first",
        observed=True
    )
    .reset_index()
)

metric_cols_present = [c for c in metric_en_in_order if c in fin_panel.columns]
fin_panel = fin_panel[KEY + ["year"] + metric_cols_present]

# --------------------------------------------------
# 6. Static descriptors
#    IMPORTANT: keep rank_2019 ... rank_2024 here
# --------------------------------------------------

id_map = {
    "Firma": "company",
    "Miejscowość": "city",
    "GPW": "gpw",
    "Rok założenia (KRS)": "incorporation_year_krs",
    "Rok założenia biznesu": "business_start_year",
    "REGON": "regon",
    "KRS": "krs",
    "NIP": "nip",
    "Forma prawna": "legal_form",
    "PK": "pkd",
    "Opis": "pkd_description",
    "Lista200": "sector",
    "Rodzaj właściciela": "owner_type",
    "S/J": "sj",
}

# rank columns stay in static_cols
static_cols = [c for c in df.columns if c not in value_cols]

static = df[static_cols].drop_duplicates(subset=["NIP"], keep="first").copy()
static = static.rename(columns={c: id_map.get(c, slugify(c)) for c in static.columns})

# --------------------------------------------------
# 7. Merge into final panel
# --------------------------------------------------

df_panel = fin_panel.merge(static, left_on="NIP", right_on="nip", how="left")

# Drop original Polish key
df_panel = df_panel.drop(columns=["NIP"])

# --------------------------------------------------
# 8. Reorder columns
# --------------------------------------------------

base = ["nip", "company", "year"]

rank_cols_en = [c for c in static.columns if c.startswith("rank_")]
static_en = [c for c in static.columns if c not in ["nip", "company"] + rank_cols_en]

df_panel = df_panel[base + rank_cols_en + static_en + metric_cols_present]

# --------------------------------------------------
# 9. Fix data types
# --------------------------------------------------

df_panel["year"] = pd.to_numeric(df_panel["year"], errors="coerce").astype("Int64")

for c in rank_cols_en:
    df_panel[c] = pd.to_numeric(df_panel[c], errors="coerce").round().astype("Int64")

# Optional: ensure nip stays as text
df_panel["nip"] = df_panel["nip"].astype(str).str.strip()

# --------------------------------------------------
# 10. Diagnostics
# --------------------------------------------------

print("Panel shape:", df_panel.shape)

print("\nRank sample:")
sample_cols = ["nip", "year"] + rank_cols_en[:3]
print(df_panel[sample_cols].head(10))

# --------------------------------------------------
# 11. Save outputs
# --------------------------------------------------

df_panel.to_excel(OUTPUT_XLSX, index=False)
df_panel.to_parquet(OUTPUT_PARQUET, index=False)

print("\nSaved Excel:", OUTPUT_XLSX)
print("Saved Parquet:", OUTPUT_PARQUET)

Panel shape: (14694, 35)

Rank sample:
          nip  year  rank_2024  rank_2023  rank_2022
0  1010001065  2019        670        403        291
1  1010001065  2020        670        403        291
2  1010001065  2021        670        403        291
3  1010001065  2022        670        403        291
4  1010001065  2023        670        403        291
5  1010001065  2024        670        403        291
6  1010002219  2019       <NA>       <NA>       <NA>
7  1010002219  2020       <NA>       <NA>       <NA>
8  1010002219  2021       <NA>       <NA>       <NA>
9  1010002219  2022       <NA>       <NA>       <NA>

Saved Excel: Data_panel_2019-2024.xlsx
Saved Parquet: ../2026/Data_panel_2019-2024.parquet


# CLEAN-UP

In [3]:
# -------------------------------------------------------
# Duplicates
# -------------------------------------------------------

# Step 1: Identify duplicates based on 'nip' and 'year'
duplicates = df_panel[df_panel.duplicated(subset=['nip', 'year'], keep=False)]

# Step 2: Display duplicates for inspection
print("Duplicate rows based on 'nip' and 'year':")
print(duplicates[['nip', 'company', 'year']])

Duplicate rows based on 'nip' and 'year':
Empty DataFrame
Columns: [nip, company, year]
Index: []


In [4]:
# -------------------------------------------------------
# MANUAL REMOVALS
# -------------------------------------------------------
manually_remove = ["7740001454","5260152146B","5262297860B","7521457964B"]
df_panel = df_panel[~df_panel["nip"].isin(manually_remove)]



In [5]:
# -------------------------------------------------------
# Missing data report
# -------------------------------------------------------


financial_cols = [
    'sales','operating_result','profit_before_tax','net_profit',
    'depreciation','exports','employment','wages_total',
    'total_assets','fixed_assets','current_assets','equity'
]

missing_by_year = (
    df_panel.groupby('year')[financial_cols]
      .apply(lambda x: x.isna().sum())
)

display(missing_by_year)

# Companies with at least one missing financial value

companies_with_missing = (
    df_panel.groupby('nip')[financial_cols]
      .apply(lambda x: x.isna().any().any())
)

num_companies_with_missing = companies_with_missing.sum()

print("Companies with at least one missing financial value:", num_companies_with_missing)

total_companies = df_panel['nip'].nunique()

print("Total companies:", total_companies)
print("Companies with missing data:", num_companies_with_missing)
print("Companies with complete financial data:", total_companies - num_companies_with_missing)

# -------------------------------------------------------
# PANEL COVERAGE DIAGNOSTIC
# -------------------------------------------------------

years = sorted(df_panel["year"].unique())
all_firms = set(df_panel["nip"])
total_firms = len(all_firms)

summary_rows = []
missing_by_year = {}

for y in years:
    firms_in_year = set(df_panel.loc[df_panel["year"] == y, "nip"])
    
    present = len(firms_in_year)
    missing = sorted(all_firms - firms_in_year)
    
    missing_by_year[y] = missing
    
    summary_rows.append({
        "year": y,
        "firms_present": present,
        "firms_missing": len(missing),
        "total_firms": total_firms
    })

panel_summary = pd.DataFrame(summary_rows)

print("Panel coverage summary")
display(panel_summary)

# -------------------------------------------------------
# MISSING COMPANIES (ALL YEARS)
# -------------------------------------------------------

missing_rows = []

for y, firms in missing_by_year.items():
    for nip in firms:
        company = df_panel.loc[df_panel["nip"] == nip, "company"].iloc[0]
        
        missing_rows.append({
            "year_missing": y,
            "nip": nip,
            "company": company
        })

missing_companies = pd.DataFrame(missing_rows)

print("Companies missing from specific years")
display(missing_companies)

# -------------------------------------------------------
# ARE MISSING FIRMS SUBSETS OR NEW?
# -------------------------------------------------------

baseline_missing = set(missing_by_year.get(years[0], []))

subset_check = []

for y in years:
    missing_set = set(missing_by_year[y])
    
    subset_check.append({
        "year": y,
        "missing_firms": len(missing_set),
        "subset_of_baseline_missing": missing_set.issubset(baseline_missing)
    })

subset_table = pd.DataFrame(subset_check)

print("Are missing firms subsets of baseline missing?")
display(subset_table)

# -------------------------------------------------------
# REMOVE COMPANIES WITH MISSING YEARS?
# -------------------------------------------------------

years_required = df_panel["year"].nunique()

years_per_firm = df_panel.groupby("nip")["year"].nunique()

full_coverage_firms = years_per_firm[years_per_firm == years_required].index

df_panel = df_panel[df_panel["nip"].isin(full_coverage_firms)]
print("Unique companies:", df_panel["nip"].nunique())

# -------------------------------------------------------
# REMOVE FIRMS WITH ZERO OR MISSING SALES IN 2019
# -------------------------------------------------------

sales_2019 = df_panel.loc[df_panel["year"] == 2019, ["nip", "sales"]]

valid_sales_firms = sales_2019.loc[sales_2019["sales"] > 0, "nip"]

df_panel = df_panel[df_panel["nip"].isin(valid_sales_firms)].copy()

print("Companies after removing zero/missing sales in 2019:", df_panel["nip"].nunique())


,sales,operating_result,profit_before_tax,net_profit,depreciation,exports,employment,wages_total,total_assets,fixed_assets,current_assets,equity
year,,,,,,,,,,,,
2019,0,8,8,7,14,25,36,21,9,10,10,9
2020,0,7,7,7,9,23,36,14,7,8,8,7
2021,0,8,8,8,11,19,28,14,8,9,9,8
2022,0,9,8,7,11,18,23,13,8,9,9,9
2023,0,11,10,9,12,15,23,13,10,13,13,11
2024,0,14,13,12,17,23,25,19,13,17,17,14


Companies with at least one missing financial value: 85
Total companies: 2450
Companies with missing data: 85
Companies with complete financial data: 2365
Panel coverage summary


,year,firms_present,firms_missing,total_firms
0,2019,2442,8,2450
1,2020,2445,5,2450
2,2021,2445,5,2450
3,2022,2447,3,2450
4,2023,2450,0,2450
5,2024,2450,0,2450


Companies missing from specific years


,year_missing,nip,company
0,2019,1251739335,"Gołębiewski Holding sp. z o.o., Radzymin"
1,2019,5252837556,"Microsoft Poland Operations sp. z o.o., Warszawa"
2,2019,5272605317,"Bidcorp Poland sp. z o.o. GK, Szczecin"
3,2019,5730003999,"Ergis SA GK, Warszawa"
4,2019,6412479943,"3W SA GK, Ruda Śląska"
5,2019,7341001487,"Grupa FAKRO, Nowy Sącz"
6,2019,7351027264,"Firma Kojs, Jabłonka"
7,2019,9492262904,"ZF Passive Safety Systems Poland sp. z o.o., C..."
8,2020,1251739335,"Gołębiewski Holding sp. z o.o., Radzymin"
9,2020,5252837556,"Microsoft Poland Operations sp. z o.o., Warszawa"


Are missing firms subsets of baseline missing?


,year,missing_firms,subset_of_baseline_missing
0,2019,8,True
1,2020,5,True
2,2021,5,False
3,2022,3,False
4,2023,0,True
5,2024,0,True


Unique companies: 2440
Companies after removing zero/missing sales in 2019: 2352


In [6]:
# -------------------------------------------------------
# DIAGNOSTICS COMPARE WIDE AND PANEL
# -------------------------------------------------------

nips_wide = set(df_wide["NIP"].astype(str).str.strip())
nips_panel = set(df_panel["nip"].astype(str).str.strip())

dropped = sorted([x for x in (nips_wide - nips_panel) if x not in ["", "nan", "None"]])

print("Unique NIP in wide:", len(nips_wide))
print("Unique nip in panel:", len(nips_panel))
print("Dropped NIPs (non-empty):", len(dropped))
print(dropped[:30])

# Check blank NIP rows in the WIDE dataset
blank_nip_rows = df_wide["NIP"].isna() | (df_wide["NIP"].astype(str).str.strip() == "")
print("Rows with blank NIP in wide:", int(blank_nip_rows.sum()))

if blank_nip_rows.any():
    print(df_wide.loc[blank_nip_rows, ["Firma", "REGON", "KRS", "NIP"]].head(20))

blank_nip_panel = df_panel["nip"].isna() | (df_panel["nip"].astype(str).str.strip() == "")
print("Rows with blank nip in panel:", int(blank_nip_panel.sum()))

Unique NIP in wide: 2454
Unique nip in panel: 2352
Dropped NIPs (non-empty): 102
['1130017933', '1133003794', '1231467307', '1231495522', '1251739335', '5130268790', '5213951377', '5213956475', '5223234571', '5242435833', '5252140453', '5252609571', '5252728815', '5252766454', '5252767666', '5252773158', '5252776257', '5252801825', '5252803818', '5252806478', '5252810445', '5252813840', '5252836864', '5252837556', '5252838515', '5252873954', '5252929353', '5260015900', '5260152146B', '5262297860B']
Rows with blank NIP in wide: 0
Rows with blank nip in panel: 0


In [7]:
df_panel.to_excel(OUTPUT_XLSX, index=False)
df_panel.to_parquet(OUTPUT_PARQUET, index=False)
print("Saved files")

Saved files


In [8]:
# -------------------------------------------------------
# FINANCIAL SANITY CHECKS
# -------------------------------------------------------

checks = {}

# Exports larger than sales
checks["exports_gt_sales"] = df_panel[
    (df_panel["exports"] > df_panel["sales"]) &
    df_panel["exports"].notna() &
    df_panel["sales"].notna()
]

# Negative employment
checks["negative_employment"] = df_panel[
    (df_panel["employment"] < 0) &
    df_panel["employment"].notna()
]

# Negative assets
checks["negative_assets"] = df_panel[
    (df_panel["total_assets"] < 0) &
    df_panel["total_assets"].notna()
]

# Negative wages
checks["negative_wages"] = df_panel[
    (df_panel["wages_total"] < 0) &
    df_panel["wages_total"].notna()
]

# Equity larger than assets (balance sheet inconsistency)
checks["equity_gt_assets"] = df_panel[
    (df_panel["equity"] > df_panel["total_assets"]) &
    df_panel["equity"].notna() &
    df_panel["total_assets"].notna()
]

# Current assets larger than total assets
checks["current_assets_gt_total"] = df_panel[
    (df_panel["current_assets"] > df_panel["total_assets"]) &
    df_panel["current_assets"].notna() &
    df_panel["total_assets"].notna()
]

# Fixed assets larger than total assets
checks["fixed_assets_gt_total"] = df_panel[
    (df_panel["fixed_assets"] > df_panel["total_assets"]) &
    df_panel["fixed_assets"].notna() &
    df_panel["total_assets"].notna()
]


# -------------------------------------------------------
# REPORT RESULTS
# -------------------------------------------------------

summary = []

for name, df_check in checks.items():
    
    summary.append({
        "check": name,
        "observations": len(df_check),
        "firms": df_check["nip"].nunique()
    })

sanity_summary = pd.DataFrame(summary)

print("Financial sanity check summary")
display(sanity_summary)

Financial sanity check summary


,check,observations,firms
0,exports_gt_sales,95,68
1,negative_employment,0,0
2,negative_assets,0,0
3,negative_wages,0,0
4,equity_gt_assets,7,5
5,current_assets_gt_total,3,3
6,fixed_assets_gt_total,1,1


In [9]:
display(checks["exports_gt_sales"][["nip","company","year","sales","exports"]])

display(checks["equity_gt_assets"][["nip", "company", "year", "equity", "total_assets"]])

,nip,company,year,sales,exports
21,1050000072,"Gillette Poland International sp. z o.o., Łódź",2022,5.522325e+05,552233.0
104,1090001066,"Katcon Polska sp. z o.o., Błonie",2021,2.339578e+06,2339689.0
431,1181363444,"Rockwell Automation sp. z o.o., Warszawa",2019,5.555590e+05,555736.0
436,1181363444,"Rockwell Automation sp. z o.o., Warszawa",2024,9.387100e+05,939600.0
639,2220009234,"Manuli Hydraulics Polska SA, Mysłowice",2020,4.588410e+05,459852.0
...,...,...,...,...,...
14237,9542352817,"HUF Polska sp. z o.o., Tychy",2024,6.513640e+05,656785.0
14304,9542765577,"Primeore Trading (Polska) sp. z o.o., Katowice",2019,1.363828e+06,1367157.0
14307,9542765577,"Primeore Trading (Polska) sp. z o.o., Katowice",2022,2.052117e+05,211006.0
14411,9570752316,"Intel Technology Poland sp. z o.o., Gdańsk",2024,1.201413e+06,1201413.0


,nip,company,year,equity,total_assets
594,1251649589,"Cygan sp. z o.o. sp.k., Marki",2020,1.680550e+05,168054.00000
4418,5291732502,"Tech Route sp. z o.o., Grodzisk Mazowiecki",2022,3.539136e+04,11702.60909
4419,5291732502,"Tech Route sp. z o.o., Grodzisk Mazowiecki",2023,1.367845e+05,22781.15437
4420,5291732502,"Tech Route sp. z o.o., Grodzisk Mazowiecki",2024,3.606887e+05,25047.62516
8036,6772001669,"Ordipol sp. z o.o. (w upadłości), Bielany Wroc...",2022,4.643000e+04,46429.00000
8655,6991938413,"Gobarto Hodowca sp. z o.o., Warszawa",2023,4.476063e+04,293.00000
13179,8982196752,"Ten Square Games SA GK, Wrocław",2024,2.559688e+06,426674.00000
